# 錯誤更正碼 1012

## Reed–Solomon Codes

RS codes 是實務上最常用的 algebraic codes 之一，從 CD、QR code 到深空通訊都看得到它。

食譜
- 用多項式取值定義 RS code，得到 $[n, k, n - k + 1]_q$。
- Singleton bound 說這已是最大的 minimum distance，也就是 MDS。
- 用 Lagrange interpolation 補回 erasures。
- 用 Berlekamp–Welch algorithm 更正 errors。
- 用 primitive element 排列座標，得到 generator polynomial 與 syndrome decoding。
- 在 $\mathbb F_{256}$ 上，一個 symbol 就是一個 byte，可以對抗 burst errors。

### 用多項式的取值當作 codeword

固定 $\mathbb F_q$ 中 $n$ 個不同的元素 $a_1, \ldots, a_n$，稱為 **evaluation points（取值點）**。  
令 $1 \le k \le n$，並令
$$\mathbb F_q[x]_{<k} = \{f \in \mathbb F_q[x] : f = 0\text{ 或 }\deg f < k\}.$$
它是 $\mathbb F_q$ 上的 $k$ 維向量空間，基底是 $1, x, \ldots, x^{k - 1}$。

**Reed–Solomon code** 是所有取值向量形成的集合：
$$\mathrm{RS}_k(a_1, \ldots, a_n) = \{(f(a_1), \ldots, f(a_n)) : f \in \mathbb F_q[x]_{<k}\}.$$
取值映射 $\operatorname{ev}(f) = (f(a_1), \ldots, f(a_n))$ 是線性的，所以 RS code 是 linear code。  
取值點必須互不相同，所以 $n \le q$。

資訊 $u = (u_0, \ldots, u_{k - 1}) \in \mathbb F_q^k$ 對應到多項式
$$f_u(x) = u_0 + u_1x + \cdots + u_{k - 1}x^{k - 1}.$$
編碼就是在每個取值點計算 $f_u$。  
傳送的是 $n$ 個取值，而不是 $k$ 個係數；多出來的 $n - k$ 個取值就是 redundancy。

次數小於 $k$ 的多項式，由任意 $k$ 個點上的取值決定。  
例如 $k = 2$ 時，$f_u$ 是一條直線。  
傳送直線上的 $n$ 個點，只要收到其中任意兩個正確的點，就能畫回整條直線。

### Dimension 與 minimum distance

若 $f \ne 0$ 且 $\deg f < k$，root bound 說 $f$ 至多有 $k - 1$ 個根。  
所以 $f(a_1), \ldots, f(a_n)$ 之中至多有 $k - 1$ 個零，這導致
$$\operatorname{wt}(\operatorname{ev}(f)) \ge n - k + 1 \ge 1.$$
因此 $\operatorname{ev}$ 是一一的，code 的 dimension 是 $k$，minimum distance 至少是 $n - k + 1$。

另一方面，多項式
$$f(x) = (x - a_1)(x - a_2) \cdots (x - a_{k - 1})$$
的次數是 $k - 1$，在前 $k - 1$ 個取值點為零，在其他 $n - k + 1$ 個取值點都不為零。  

所以 minimum distance 恰好是 $n - k + 1$：
$$\boxed{\mathrm{RS}_k(a_1, \ldots, a_n)\text{ 是 }[n, k, n - k + 1]_q\text{ code}.}$$
$k = 1$ 時，空乘積取為 $1$，code 就是 repetition code。

### $q$-ary Singleton bound 與 MDS codes

0907 講義的 Singleton bound 對任意大小的 alphabet 都成立。  
設 $\mathcal C \subseteq \mathcal A^n$ 是 minimum distance 為 $d$ 的 block code，$|\mathcal A| = q$。  
刪去每個 codeword 的最後 $d - 1$ 個 symbols，不同的 codewords 仍然不同，否則它們的距離至多是 $d - 1$。  
因此
$$|\mathcal C| \le q^{n - d + 1}.$$
對 $[n, k, d]_q$ linear code，$|\mathcal C| = q^k$，所以 $d \le n - k + 1$。

等號成立的 code 稱為 **MDS code（maximum distance separable code）**。  
RS code 達到等號：固定 $n$ 與 $k$ 時，沒有任何 code 的 minimum distance 比它更大。  
代價是 $n \le q$：要做長的 RS code，就需要大的有限體。

和之前的 codes 比較：

| Code | 參數 | $n - k + 1$ | MDS |
|---|---|---|---|
| Repetition | $[n, 1, n]_q$ | $n$ | 是 |
| Single parity-check | $[n, n - 1, 2]_q$ | $2$ | 是 |
| Hamming | $[7, 4, 3]_2$ | $4$ | 否 |
| Extended Hamming | $[8, 4, 4]_2$ | $5$ | 否 |
| $\mathbb F_{256}$ 上的 RS | $[255, 223, 33]_{256}$ | $33$ | 是 |



Binary 的 MDS codes 只有 repetition、single parity-check 與 $\mathbb F_2^n$ 本身；  
有趣的 MDS codes 需要較大的 $q$。

### Generator matrix 是 Vandermonde matrix

沿用 row-vector 慣例，如果想要 $\operatorname{ev}(f_u) = uG$，就必須
$$G = \begin{pmatrix}
1 & 1 & \cdots & 1 \\
a_1 & a_2 & \cdots & a_n \\
\vdots & \vdots & & \vdots \\
a_1^{k - 1} & a_2^{k - 1} & \cdots & a_n^{k - 1}
\end{pmatrix}.$$
第 $i$ row 是 monomial $x^i$ 的取值向量，$i = 0, \ldots, k - 1$；  
第 $j$ column 是取值點 $a_j$ 的各個冪次。  
這種矩陣稱為 **Vandermonde matrix**。

### 例子：$\mathbb F_7$ 上的 $[6, 2, 5]_7$ code

這個例子會一直用到後面的 syndrome decoding。

在 $\mathbb F_7$ 中，$3$ 是 primitive element：
$$3^0 = 1,\quad 3^1 = 3,\quad 3^2 = 2,\quad 3^3 = 6,\quad 3^4 = 4,\quad 3^5 = 5,\quad 3^6 = 1.$$
令 $\alpha = 3$，座標編號為 $0, 1, \ldots, 5$，座標 $i$ 的取值點是 $\alpha^i$：
$$(\alpha^0, \alpha^1, \ldots, \alpha^5) = (1, 3, 2, 6, 4, 5).$$
取 $k = 2$，得到 $d = 6 - 2 + 1 = 5$ 的 RS code，可以更正 $2$ 個 errors。  


Generator matrix 為
$$G = \begin{pmatrix}
1 & 1 & 1 & 1 & 1 & 1 \\
1 & 3 & 2 & 6 & 4 & 5
\end{pmatrix}.$$
取資訊 $u = (3, 2)$，也就是 $f(x) = 3 + 2x$，得到
$$c = uG = (5, 2, 0, 1, 4, 6).$$
例如座標 $1$ 是 $f(3) = 9 = 2$，座標 $2$ 是 $f(2) = 7 = 0$。  
這個 codeword 的 weight 是 $5$，恰好等於 minimum distance。

下面用電腦列出全部 $7^2 = 49$ 個 codewords 的 weights。  
這份講義的程式碼使用 SageMath kernel：`GF(7)` 是 $\mathbb F_7$，`PolynomialRing` 建立多項式環。  
SageMath 會把 `^` 解讀成次方，而不是 Python 的 XOR。  
但是為了減少出 bug 的機率，這邊一律用 `**` 表示次方。

In [1]:
F = GF(7)
R = PolynomialRing(F, "x")
x = R.gen()
alpha = F(3)
n, k = 6, 2
points = [alpha**i for i in range(n)]

def encode(f):
    return vector(F, [f(a) for a in points])

print(points)
print(encode(3 + 2*x))

[1, 3, 2, 6, 4, 5]
(5, 2, 0, 1, 4, 6)


In [2]:
from collections import Counter
from itertools import product

weights = Counter(encode(R(list(u))).hamming_weight() for u in product(F, repeat=k))
print(sorted(weights.items()))

[(0, 1), (5, 36), (6, 12)]


除了零向量，每個 codeword 的 weight 都是 $5$ 或 $6$。  
Weight 為 $5$ 的有 $36$ 個：  
$u_0, u_1$ 都不為零時，$f_u$ 的根 $-u_0/u_1$ 是某個非零元素，恰好是一個取值點。  
Weight 為 $6$ 的有 $12$ 個：  
非零常數 $u_0$，以及根為 $0$ 的 $u_1x$；$0$ 不是取值點。

## Erasure Decoding

### Lagrange interpolation

給定 $k$ 個不同的點 $b_1, \ldots, b_k$ 與任意的值 $y_1, \ldots, y_k$，定義
$$L_i(x) = \prod_{j \ne i}\frac{x - b_j}{b_i - b_j},\qquad i = 1, \ldots, k.$$
$L_i$ 的次數是 $k - 1$，滿足 $L_i(b_i) = 1$，且 $L_i(b_j) = 0$（for all $j \ne i$）。  


因此
$$f(x) = \sum_{i = 1}^k y_i L_i(x)$$
是次數小於 $k$ 且滿足 $f(b_i) = y_i$ 的多項式。  
由 root bound，這樣的多項式唯一：兩個解的差次數小於 $k$，卻有 $k$ 個根。

### 補回 erasures

在 erasure channel 上，有些 symbols 變成「?」，但沒有被擦除的 symbols 都正確。  
若 RS codeword 至多有 $n - k$ 個 erasures，就至少剩下 $k$ 個正確的取值。  
任選其中 $k$ 個，用 Lagrange interpolation 求出 $f$，再重新取值，就得到完整的 codeword。  
所以 RS code 能補回任意 $n - k = d - 1$ 個 erasures。

這也是任何 $[n, k]_q$ code 的極限。  
若擦除超過 $n - k$ 個 symbols，剩下的不到 $k$ 個 symbols 只有不到 $q^k$ 種可能，無法區分 $q^k$ 個 codewords。

例如在 $[6, 2, 5]_7$ code 中，收到
$$(5, ?, ?, ?, 4, ?).$$
剩下的兩個取值是 $f(1) = 5$ 與 $f(4) = 4$。  
直線的斜率是 $(4 - 5)/(4 - 1) = -3^{-1} = -5 = 2$，所以
$$f(x) = 5 + 2(x - 1) = 3 + 2x.$$
重新取值得到 $c = (5, 2, 0, 1, 4, 6)$，四個 erasures 全部補回。

In [3]:
f = R.lagrange_polynomial([(F(1), F(5)), (F(4), F(4))])
print(f)
print(encode(f))

2*x + 3
(5, 2, 0, 1, 4, 6)


### 應用：分散式儲存

資料中心常把一個檔案切成 $k$ 塊，再額外算出 $n - k$ 塊 parity，分別存到 $n$ 台機器。  
對每個 byte 的位置，$k$ 塊資料與 $n - k$ 塊 parity 在該位置的 bytes 組成一個 RS codeword。  
任意 $n - k$ 台機器故障時，剩下的任意 $k$ 台就能還原整個檔案。

| 做法 | 儲存空間 | 保證可容忍的故障數 |
|---|---|---|
| 複製三份 | $3$ 倍 | $2$ |
| $k = 6$、$n = 9$ 的 RS code | $1.5$ 倍 | $3$ |

### Errors 與 erasures 同時出現

若已知 $s$ 個 erasures 的位置，直接刪掉這些座標。  
剩下的座標仍是不同取值點上的 RS code，參數為
$$[n - s, k, n - s - k + 1]_q.$$
所以只要
$$2e + s \le n - k = d - 1,$$
就還能更正 $e$ 個 errors。  
每個 erasure 只消耗一個 redundancy symbol，每個 error 卻要消耗兩個，  
這是因為 error 不但值錯，連位置都不知道，總共兩個不知道。

## MDS Codes

### 任意 $k$ 個 columns 都線性獨立

令 $\mathcal C$ 是 $[n, k]_q$ linear code，$G$ 是 rank 為 $k$ 的 $k \times n$ generator matrix。  
對大小為 $k$ 的座標集合 $S$，令 $G_S$ 是只保留 $S$ 中 columns 的 $k \times k$ 子矩陣。  
因為 $u \mapsto uG$ 是一一的，
$$\text{有非零 codeword 在 }S\text{ 上全為零}
\quad\Longleftrightarrow\quad
\text{有 }u \ne 0\text{ 使 }uG_S = 0
\quad\Longleftrightarrow\quad
G_S\text{ 不可逆}.$$

非零 codeword 的 weight 至多為 $n - k$，等價於它在某 $k$ 個座標上全為零。  
因此
$$\boxed{\mathcal C\text{ 是 MDS}\quad\Longleftrightarrow\quad G\text{ 的任意 }k\text{ 個 columns 都線性獨立}.}$$
換句話說，任意 $k$ 個座標都是 **information set**：這 $k$ 個 symbols 唯一決定整個 codeword。  
前面的 erasure decoding 就是在用這個性質。

### Vandermonde determinant

在 RS code 的 $G$ 中任選 $k$ 個 columns，得到 $k \times k$ 的 Vandermonde matrix
$$V(b_1, \ldots, b_k) = \begin{pmatrix}
1 & \cdots & 1 \\
b_1 & \cdots & b_k \\
\vdots & & \vdots \\
b_1^{k - 1} & \cdots & b_k^{k - 1}
\end{pmatrix},\qquad b_i\text{ 互不相同}.$$
若 row vector $w$ 滿足 $wV = 0$，則係數為 $w$ 的多項式在 $b_1, \ldots, b_k$ 都為零。  
它的次數小於 $k$，由 root bound 必為零多項式，所以 $V$ 可逆；這是 RS code 為 MDS 的另一種說法。

行列式的 explicit formula：
$$\det V(b_1, \ldots, b_k) = \prod_{1 \le i < j \le k}(b_j - b_i).$$
證明：把 $b_k$ 換成變數 $x$，沿最後一個 column 展開，  
得到次數至多 $k - 1$ 的多項式，$x^{k - 1}$ 的係數是 $\det V(b_1, \ldots, b_{k - 1})$。  
它在 $x = b_1, \ldots, b_{k - 1}$ 為零，因為此時有兩個相同的 columns。  
所以它等於 $\det V(b_1, \ldots, b_{k - 1})\prod_{i < k}(x - b_i)$，再對 $k$ 做歸納。

### Uniform matroid 與 weight distribution

用 0921 講義的 matroid 語言，MDS 表示 column matroid 的 bases 恰好是所有大小為 $k$ 的子集合。  
這個 matroid 稱為 **uniform matroid** $U_{k, n}$。  
Dual matroid 的 bases 是這些子集合的補集合，也就是所有大小為 $n - k$ 的子集合，即 $U_{n - k, n}$。  
$\mathcal C^\bot$ 的 column matroid 是 dual matroid，所以 MDS code 的 dual 也是 MDS，參數為 $[n, n - k, k + 1]_q$。

Greene theorem 用 column matroid 的 Tutte polynomial 算出 weight enumerator。  
$U_{k, n}$ 只和 $n, k$ 有關，所以 MDS code 的 weight distribution 只由 $n, k, q$ 決定。  
令 $d = n - k + 1$，weight 為 $w$ 的 codewords 個數是
$$A_w = \binom nw\sum_{j = 0}^{w - d}(-1)^j\binom wj\bigl(q^{w - d + 1 - j} - 1\bigr),\qquad d \le w \le n.$$
例如 $[6, 2, 5]_7$ 有 $A_5 = 6 \cdot 6 = 36$、$A_6 = (7^2 - 1) - 6 \cdot 6 = 12$，和前面的計算相同。

### Dual code 是 generalized RS code

給定非零的 $v_1, \ldots, v_n \in \mathbb F_q$，  
令 $a = (a_1, \ldots, a_n)$、$v = (v_1, \ldots, v_n)$，  
定義 **generalized Reed–Solomon code**
$$\mathrm{GRS}_k(a, v) = \{(v_1 f(a_1), \ldots, v_n f(a_n)) : f \in \mathbb F_q[x]_{<k}\}.$$
每個座標乘上非零常數不改變 weight，所以它仍是 $[n, k, n - k + 1]_q$ MDS code。  
$v_1 = \cdots = v_n = 1$ 就是原本的 RS code，記為 $\mathrm{RS}_k(a)$。

為了找到 RS 的 dual，令
$$v_i = \prod_{j \ne i}\frac{1}{a_i - a_j}.$$
對次數至多 $n - 2$ 的多項式 $h$，用全部 $n$ 個取值點做 Lagrange interpolation，會得到 $h$ 本身。  
比較兩側 $x^{n - 1}$ 的係數，得到
$$\sum_{i = 1}^n v_i h(a_i) = 0.$$
取 $h = fg$，其中 $\deg f < k$、$\deg g < n - k$，就得到 $\operatorname{ev}(f) \cdot (v_1 g(a_1), \ldots, v_n g(a_n)) = 0$。

$\mathrm{GRS}_{n - k}(a, v)$ 的 dimension 是 $n - k$，與 $\mathrm{RS}_k(a)^\bot$ 相同，所以
$$\mathrm{RS}_k(a)^\bot = \mathrm{GRS}_{n - k}(a, v).$$
因此 RS code 的 parity-check matrix 也是 Vandermonde 型的矩陣，只是每個 row 多乘上 $v_i$：
$$H = \bigl(v_i a_i^j\bigr)_{1 \le i \le n,\ 0 \le j \le n - k - 1}.$$

## Berlekamp–Welch Decoding

### 能更正幾個 errors？

Minimum distance $d = n - k + 1$ 保證可以更正
$$t = \left\lfloor\frac{n - k}{2}\right\rfloor$$
個 errors。  
困難在於怎麼找到最近的 codeword。  
直接比較 $q^k$ 個 codewords，或對 $\binom nk$ 種 $k$ 個座標的組合分別做 interpolation，都太慢。  
例如 $[255, 223, 33]_{256}$ 有 $256^{223}$ 個 codewords，$\binom{255}{223}$ 也超過 $10^{40}$。

接下來介紹兩種多項式時間的 decoders：
- Berlekamp–Welch：直接從取值解出 $f$，適用於任意的取值點。
- Syndrome decoding：只用 $n - k$ 個 syndromes 找出 error vector，使用 cyclic RS codes 的結構。

### Error locator polynomial

傳送 $c = \operatorname{ev}(f)$，收到 $y = c + e$。  
令錯誤位置的集合為 $B = \{i : e_i \ne 0\}$，並假設 $|B| \le t$。  
定義 **error locator polynomial**
$$E(x) = \prod_{i \in B}(x - a_i).$$
它在錯誤位置為零，在其他取值點都不為零；decoder 一開始不知道 $E$，也不知道 $f$。

對每個座標 $i$，下式都成立：
$$y_i E(a_i) = f(a_i) E(a_i).$$
若 $i \notin B$，則 $y_i = f(a_i)$；若 $i \in B$，則兩側都是 $0$。  
這個方程含有兩個未知多項式的乘積 $fE$，不是線性的。

### 把未知的乘積藏進 $Q$

實際錯誤少於 $t$ 個時，把 $E$ 乘上 $x^{t - |B|}$，上式仍然成立；所以可以假設 $E$ 是 $t$ 次 monic polynomial。  
令 $Q(x) = f(x) E(x)$，它的次數小於 $k + t$。  
方程變成
$$y_i E(a_i) = Q(a_i),\qquad i = 1, \ldots, n,$$
對 $E$ 與 $Q$ 的係數是**線性**的。

未知數是
$$E(x) = x^t + E_{t - 1}x^{t - 1} + \cdots + E_0,\qquad
Q(x) = Q_0 + Q_1x + \cdots + Q_{k + t - 1}x^{k + t - 1}.$$
共有 $k + 2t \le n$ 個未知數與 $n$ 個方程。

**定理：** 若 $|B| \le t$，則方程組有解，而且任意一組解 $(E_1, Q_1)$ 都滿足 $Q_1 = fE_1$。

證明：令 $(E, Q)$ 是由真正的 $f$ 得到的解，考慮
$$P(x) = Q_1(x) E(x) - Q(x) E_1(x).$$
它的次數至多是 $k + 2t - 1 \le n - 1$。  
在每個取值點，$P(a_i) = y_i E_1(a_i) E(a_i) - y_i E(a_i) E_1(a_i) = 0$。  
$P$ 有 $n$ 個根，次數卻小於 $n$，所以 $P = 0$。  
於是 $Q_1E = QE_1 = fEE_1$，消去 $E$ 便得到 $Q_1 = fE_1$。

Berlekamp–Welch decoder 的步驟：
1. 解 $n$ 個線性方程，得到任意一組 $(E, Q)$。
2. 做多項式除法，得到 $f = Q/E$。
3. 輸出 $f$，或重新取值得到 codeword $\operatorname{ev}(f)$。

若方程組無解，或 $E$ 不整除 $Q$，就表示錯誤超過 $t$ 個，宣告解碼失敗。  
反過來，若 $E$ 整除 $Q$，則 $y_i = f(a_i)$ 對每個 $E(a_i) \ne 0$ 的 $i$ 都成立，所以輸出與 $y$ 至多相差 $t$ 個位置。

### 例子：更正兩個 errors

沿用 $c = (5, 2, 0, 1, 4, 6)$，其中 $f(x) = 3 + 2x$。  
在座標 $1$ 加上 $2$、在座標 $3$ 加上 $4$，收到
$$y = (5, 4, 0, 5, 4, 6).$$
此時 $t = 2$，未知數是
$$E(x) = x^2 + E_1x + E_0,\qquad Q(x) = Q_0 + Q_1x + Q_2x^2 + Q_3x^3.$$
以座標 $0$ 為例，取值點是 $1$、$y_0 = 5$，方程為
$$Q_0 + Q_1 + Q_2 + Q_3 - 5(E_0 + E_1) = 5.$$

六個取值點各給一個方程。  
下面的程式把它們寫成 augmented matrix，未知數的順序是 $Q_0, Q_1, Q_2, Q_3, E_0, E_1$。

In [4]:
def bw_system(points, y, k, t):
    rows, rhs = [], []
    for a, b in zip(points, y):
        powers = [a**j for j in range(k + t + 1)]
        rows.append(powers[:k + t] + [-b * p for p in powers[:t]])
        rhs.append(b * powers[t])
    return matrix(rows), vector(rhs)

y = vector(F, [5, 4, 0, 5, 4, 6])
A, rhs = bw_system(points, y, k, 2)
print(A.augment(rhs))

[1 1 1 1 2 2 5]
[1 3 2 6 3 2 1]
[1 2 4 1 0 0 0]
[1 6 1 6 2 5 5]
[1 4 2 1 3 5 1]
[1 5 4 6 1 5 3]


In [5]:
solution = A.solve_right(rhs)
Q = R(list(solution[:4]))
E = R(list(solution[4:]) + [1])
print("E =", E, "=", E.factor())
print("Q =", Q)
print("(Q / E, remainder) =", Q.quo_rem(E))

E = x^2 + 5*x + 4 = (x + 1) * (x + 4)
Q = 2*x^3 + 6*x^2 + 2*x + 5
(Q / E, remainder) = (2*x + 3, 0)


解出
$$E(x) = x^2 + 5x + 4 = (x - 3)(x - 6),\qquad Q(x) = 2x^3 + 6x^2 + 2x + 5.$$
$E$ 的根是 $3 = \alpha^1$ 與 $6 = \alpha^3$，正是座標 $1$ 與 $3$；SageMath 把 $x - 6$ 與 $x - 3$ 寫成 $x + 1$ 與 $x + 4$。  
多項式除法給出 $Q(x) = (2x + 3) E(x)$，餘數為零，所以 $f(x) = 3 + 2x$，解碼成功。

手算檢查一個方程：在錯誤位置的取值點 $3$，
$$E(3) = 9 + 15 + 4 = 28 = 0,\qquad Q(3) = 54 + 54 + 6 + 5 = 119 = 0.$$
所以 $y_1 E(3) = Q(3)$ 成立，與 $y_1$ 的值無關。  
錯誤位置的方程不提供資訊，$E$ 的作用就是把它們「關掉」。

把整個流程包成一個 function。  
方程組無解時，`solve_right` 會丟出 `ValueError`，此時回傳 `None` 表示解碼失敗。

In [6]:
def bw_decode(points, y, k, t):
    A, rhs = bw_system(points, y, k, t)
    try:
        solution = A.solve_right(rhs)
    except ValueError:
        return None
    P = PolynomialRing(y.base_ring(), "x")
    Q = P(list(solution[:k + t]))
    E = P(list(solution[k + t:]) + [1])
    f, remainder = Q.quo_rem(E)
    return f if remainder == 0 else None

### 超過 $t$ 個 errors

Berlekamp–Welch 是 **bounded distance decoder**：只保證更正至多 $t$ 個 errors。  
錯誤更多時，可能宣告失敗，也可能輸出另一個與 $y$ 相差至多 $t$ 個位置的 codeword，稱為 **miscorrection**。  
下面對 $c$ 試遍全部 $\binom 63 \cdot 6^3 = 4{,}320$ 種恰好 $3$ 個 errors 的 patterns。

In [7]:
from itertools import combinations

failures = miscorrections = 0
for positions in combinations(range(n), 3):
    for values in product(F.list()[1:], repeat=3):
        e = vector(F, n)
        for i, v in zip(positions, values):
            e[i] = v
        decoded = bw_decode(points, encode(3 + 2*x) + e, k, 2)
        if decoded is None:
            failures += 1
        elif decoded != 3 + 2*x:
            miscorrections += 1
print(failures, miscorrections)

3960 360


結果是 $3{,}960$ 次宣告失敗、$360$ 次 miscorrections，沒有一次解回 $c$。  
因為 $y$ 與 $c$ 的距離是 $3 > t$，而 decoder 的輸出一定與 $y$ 相差至多 $t$ 個位置。

Miscorrections 的個數也能直接數出來。  
與 $c$ 距離為 $5$ 的 codewords $c'$ 有 $A_5 = 36$ 個。  
在 $c$ 與 $c'$ 不同的 $5$ 個座標中任選 $3$ 個，把 $c$ 在這些座標的值改成 $c'$ 的值，就落到 $c'$ 的半徑 $2$ 之內。  
共有 $36 \cdot \binom 53 = 360$ 種。

## Cyclic Reed–Solomon Codes

### 取值點是 $\alpha$ 的冪次

令 $\alpha$ 是 $\mathbb F_q$ 的 primitive element，$n = q - 1$，並把全部非零元素依序當作取值點：
$$c_i = f(\alpha^i),\qquad i = 0, 1, \ldots, n - 1.$$
這正是上一份講義最後「用 primitive element 排列座標」的順序；$[6, 2, 5]_7$ 就是 $q = 7$、$\alpha = 3$ 的情形。

此時 RS code 是 cyclic code。  
沿用上一份講義的 cyclic shift $\sigma(c)_i = c_{i - 1}$（下標取模 $n$），有
$$c_{i - 1} = f(\alpha^{i - 1}) = \widetilde f(\alpha^i),\qquad \widetilde f(x) = f(\alpha^{-1}x).$$
$\widetilde f$ 與 $f$ 的次數相同，所以 $\sigma(c)$ 仍是 codeword。

### Generator polynomial

同上一份講義，把 codeword 寫成多項式 $c(z) = c_0 + c_1z + \cdots + c_{n - 1}z^{n - 1}$。  
把 $f(x) = f_0 + f_1x + \cdots + f_{k - 1}x^{k - 1}$ 代入，對 $z = \alpha^j$ 計算：
$$c(\alpha^j) = \sum_{i = 0}^{n - 1}f(\alpha^i)\alpha^{ij}
= \sum_{m = 0}^{k - 1}f_m\sum_{i = 0}^{n - 1}\alpha^{i(m + j)}.$$
若 $\beta^n = 1$ 但 $\beta \ne 1$，等比級數給出
$$\sum_{i = 0}^{n - 1}\beta^i = \frac{\beta^n - 1}{\beta - 1} = 0.$$

當 $1 \le j \le n - k$ 且 $0 \le m \le k - 1$ 時，$1 \le m + j \le n - 1$，所以 $\alpha^{m + j} \ne 1$。  
每個內層的和都是零，因此
$$c(\alpha^j) = 0,\qquad j = 1, 2, \ldots, n - k.$$
也就是說，每個 codeword polynomial 都被下面的 **generator polynomial** 整除：
$$g(z) = (z - \alpha)(z - \alpha^2) \cdots (z - \alpha^{n - k}).$$

反過來，次數小於 $n$ 且被 $g(z)$ 整除的多項式可寫成 $g(z)m(z)$，其中 $\deg m < k$，共有 $q^k$ 個。  
RS code 也有 $q^k$ 個 codewords，而且都在這個集合中，所以兩者相同：
$$\mathrm{RS}_k(1, \alpha, \ldots, \alpha^{n - 1}) = \{c : g(z)\text{ 整除 }c(z)\}.$$
同一個 code 有兩種描述：Reed 與 Solomon（1960）的取值觀點，以及用 generator polynomial 描述 cyclic code 的觀點。

上面的計算其實給出更完整的關係：對每個 $j$，
$$c(\alpha^j) = n f_{(-j) \bmod n},$$
其中 $m \ge k$ 時 $f_m = 0$。  
這是有限體上的 **discrete Fourier transform**：codeword 在 $\alpha^j$ 的取值，就是 $f$ 的一個係數乘上 $n$。  
$n = q - 1$ 在 $\mathbb F_q$ 中等於 $-1$，所以這個常數一定可逆。  
RS code 的條件「$f_m = 0$ 對 $m \ge k$」，在 Fourier 的另一側就是「$c(\alpha^j) = 0$ 對 $1 \le j \le n - k$」。

### 例子：$[6, 2, 5]_7$ 的 generator polynomial

在 $\mathbb F_7$ 中，$\alpha = 3$，
$$g(z) = (z - 3)(z - 2)(z - 6)(z - 4) = z^4 + 6z^3 + 3z^2 + 2z + 4.$$
前面的 codeword $c = (5, 2, 0, 1, 4, 6)$ 對應
$$c(z) = 5 + 2z + z^3 + 4z^4 + 6z^5 = (6z + 3) g(z).$$
另外 $c(1) = 5 + 2 + 0 + 1 + 4 + 6 = 18 = 4$，而 $n f_0 = 6 \cdot 3 = 18 = 4$，和 Fourier 關係一致。

### Systematic encoding

實務上常希望資訊 symbols 原封不動出現在 codeword 中。  
給定資訊多項式 $m(z)$，$\deg m < k$，令 $r(z)$ 是 $z^{n - k}m(z)$ 除以 $g(z)$ 的餘數，定義
$$c(z) = z^{n - k}m(z) - r(z).$$
它被 $g(z)$ 整除，所以是 codeword。  
因為 $\deg r < n - k$，座標 $n - k, \ldots, n - 1$ 恰好是 $m$ 的係數，座標 $0, \ldots, n - k - 1$ 是 parity symbols。  
除以 $g(z)$ 可以用 shift register 實作，這是硬體 encoder 常用的方法。

例如取 $m(z) = 4 + 6z$。  
$z^4m(z) = 4z^4 + 6z^5$ 除以 $g(z)$ 的餘數是 $r(z) = 2 + 5z + 6z^3$，因此
$$c = (-2, -5, 0, -6, 4, 6) = (5, 2, 0, 1, 4, 6),$$
正是前面的 codeword。  
這不是巧合：MDS code 的任意 $k$ 個座標都是 information set，最後兩個座標 $(4, 6)$ 唯一決定 codeword。

In [8]:
Rz = PolynomialRing(F, "z")
z = Rz.gen()
g = prod(z - alpha**j for j in range(1, n - k + 1))
c_poly = Rz(list(encode(3 + 2*x)))
print("g =", g)
print("(c / g, remainder) =", c_poly.quo_rem(g))
print("r =", (z**(n - k) * (4 + 6*z)) % g)

g = z^4 + 6*z^3 + 3*z^2 + 2*z + 4
(c / g, remainder) = (6*z + 3, 0)
r = 6*z^3 + 5*z + 2


### Parity-check matrix 與 syndromes

條件 $c(\alpha^j) = 0$ 對 $c$ 是線性的。  
沿用本課的慣例，令 $H$ 是 $n \times (n - k)$ 的矩陣，row 編號 $i = 0, \ldots, n - 1$，column 編號 $j = 1, \ldots, n - k$：
$$H = \bigl(\alpha^{ij}\bigr),\qquad cH = \bigl(c(\alpha), c(\alpha^2), \ldots, c(\alpha^{n - k})\bigr).$$
這和 dual code 的 GRS 描述一致。  
因為 $\prod_{j \ne i}(x - \alpha^j) = (x^n - 1)/(x - \alpha^i)$，它在 $x = \alpha^i$ 的值是 $x^n - 1$ 的導數 $n\alpha^{-i}$。  
所以 $v_i = \alpha^i/n$，而 $v_i(\alpha^i)^{j - 1} = \alpha^{ij}/n$，和 $H$ 只差常數倍。

收到 $y = c + e$ 時，syndrome 為
$$s = yH = eH = (S_1, \ldots, S_{n - k}),\qquad S_j = y(\alpha^j) = e(\alpha^j).$$
若錯誤位置是 $i_1, \ldots, i_\nu$，錯誤值是 $Y_1, \ldots, Y_\nu$，令 $X_l = \alpha^{i_l}$，則
$$S_j = \sum_{l = 1}^\nu Y_l X_l^j,\qquad j = 1, \ldots, n - k.$$
$X_l$ 稱為 **error locators**，也就是錯誤位置的取值點。

## Syndrome Decoding

### Error locator polynomial 與線性遞迴

定義
$$\Lambda(z) = \prod_{l = 1}^\nu(1 - X_l z) = 1 + \Lambda_1z + \cdots + \Lambda_\nu z^\nu.$$
它的根是 $X_l^{-1}$。  
和 Berlekamp–Welch 的 $E(x) = \prod_l(x - X_l)$ 相比，只是係數順序反過來：$\Lambda(z) = z^\nu E(1/z)$。

把 $\Lambda(X_l^{-1}) = 0$ 乘上 $Y_l X_l^j$，得到
$$Y_l X_l^j + \Lambda_1 Y_l X_l^{j - 1} + \cdots + \Lambda_\nu Y_l X_l^{j - \nu} = 0.$$
對 $l$ 加總，就得到 syndromes 的線性遞迴：
$$S_j + \Lambda_1 S_{j - 1} + \cdots + \Lambda_\nu S_{j - \nu} = 0,\qquad j = \nu + 1, \ldots, n - k.$$
Syndromes 已知，所以這些方程對未知的 $\Lambda_1, \ldots, \Lambda_\nu$ 是線性的。

### Peterson's method

假設 $\nu \le t$，取 $j = \nu + 1, \ldots, 2\nu$ 的 $\nu$ 個方程，其中 $2\nu \le 2t \le n - k$：
$$\underbrace{\begin{pmatrix}
S_1 & S_2 & \cdots & S_\nu \\
S_2 & S_3 & \cdots & S_{\nu + 1} \\
\vdots & \vdots & & \vdots \\
S_\nu & S_{\nu + 1} & \cdots & S_{2\nu - 1}
\end{pmatrix}}_{M_\nu}
\begin{pmatrix}\Lambda_\nu \\ \Lambda_{\nu - 1} \\ \vdots \\ \Lambda_1\end{pmatrix}
= -\begin{pmatrix}S_{\nu + 1} \\ S_{\nu + 2} \\ \vdots \\ S_{2\nu}\end{pmatrix}.$$
$M_\nu$ 每條反對角線上的 entries 都相同，稱為 **Hankel matrix**。

Decoder 不知道 $\nu$，所以從 $\mu = t$ 開始檢查 $M_\mu$ 是否可逆，不可逆就把 $\mu$ 減一。  
這是可行的，因為 $M_\mu = VDV^\top$，其中
$$V = \bigl(X_l^{r - 1}\bigr)_{1 \le r \le \mu,\ 1 \le l \le \nu},\qquad
D = \operatorname{diag}(Y_1X_1, \ldots, Y_\nu X_\nu).$$
$\mu > \nu$ 時 rank 至多是 $\nu$，所以 $M_\mu$ 不可逆；$\mu = \nu$ 時 $V$ 是可逆的 Vandermonde matrix，$M_\nu$ 可逆。  
因此第一個可逆的 $M_\mu$ 恰好出現在 $\mu = \nu$。

解出 $\Lambda$ 之後：
1. **Chien search**：對每個座標 $i$ 檢查 $\Lambda(\alpha^{-i})$ 是否為零，得到錯誤位置。
2. 求錯誤值：知道錯誤位置後，就是 erasure decoding；也可以直接解 $S_j = \sum_l Y_l X_l^j$。

只有一個 error 時，$\Lambda_1 = -S_2/S_1$，因此
$$X_1 = \frac{S_2}{S_1},\qquad Y_1 = \frac{S_1^2}{S_2}.$$

### 例子：用 syndromes 更正兩個 errors

回到 $y = (5, 4, 0, 5, 4, 6)$。  
取值點依序是 $1, 3, 2, 6, 4, 5$，所以
$$S_1 = y(3) = 5 + 4 \cdot 3 + 0 \cdot 2 + 5 \cdot 6 + 4 \cdot 4 + 6 \cdot 5 = 93 = 2.$$
同樣算出
$$(S_1, S_2, S_3, S_4) = (2, 1, 1, 5).$$
$\det M_2 = S_1S_3 - S_2^2 = 2 - 1 = 1 \ne 0$，所以有兩個 errors。

解
$$\begin{pmatrix}2 & 1 \\ 1 & 1\end{pmatrix}
\begin{pmatrix}\Lambda_2 \\ \Lambda_1\end{pmatrix}
= -\begin{pmatrix}1 \\ 5\end{pmatrix}
= \begin{pmatrix}6 \\ 2\end{pmatrix}.$$
兩式相減得 $\Lambda_2 = 4$，再得 $\Lambda_1 = 2 - 4 = 5$，所以
$$\Lambda(z) = 1 + 5z + 4z^2 = (1 - 3z)(1 - 6z).$$
Error locators 是 $X_1 = 3 = \alpha^1$ 與 $X_2 = 6 = \alpha^3$，錯誤位置是座標 $1$ 與 $3$。  
這和 Berlekamp–Welch 的 $E(x) = x^2 + 5x + 4$ 係數順序相反。

錯誤值由前兩個 syndromes 決定。  
用 $X_1^2 = 9 = 2$ 與 $X_2^2 = 36 = 1$，得到
$$S_1 = 3Y_1 + 6Y_2 = 2,\qquad S_2 = 2Y_1 + Y_2 = 1.$$
解得 $Y_1 = 2$、$Y_2 = 4$，於是
$$c = y - (0, 2, 0, 4, 0, 0) = (5, 2, 0, 1, 4, 6).$$

In [9]:
y_poly = Rz(list(y))
S = [y_poly(alpha**j) for j in range(1, n - k + 1)]
M = matrix(F, [[S[0], S[1]], [S[1], S[2]]])
Lambda2, Lambda1 = M.solve_right(-vector(F, [S[2], S[3]]))
Lambda = Rz([1, Lambda1, Lambda2])
print("S =", S)
print("Lambda =", Lambda)
print("error positions:", [i for i in range(n) if Lambda(alpha**(-i)) == 0])

S = [2, 1, 1, 5]
Lambda = 4*z^2 + 5*z + 1
error positions: [1, 3]


### Key equation 與 Forney formula

令 syndrome polynomial 為 $S(z) = S_1 + S_2z + \cdots + S_{2t}z^{2t - 1}$。  
把 $S_j = \sum_l Y_l X_l^j$ 代入，對每個 $l$ 是等比級數；$1 - X_l z$ 在模 $z^{2t}$ 下可逆，所以
$$S(z) \equiv \sum_{l = 1}^\nu\frac{Y_l X_l}{1 - X_l z} \pmod{z^{2t}}.$$
乘上 $\Lambda(z)$，分母全部消去，得到 **key equation**：
$$\Lambda(z) S(z) \equiv \Omega(z) \pmod{z^{2t}},\qquad
\Omega(z) = \sum_{l = 1}^\nu Y_l X_l\prod_{m \ne l}(1 - X_m z).$$
$\Omega$ 稱為 **error evaluator polynomial**，$\deg \Omega < \nu$。

對 $\Lambda$ 取形式導數，在 $z = X_l^{-1}$ 代入時，$\Omega$ 與 $\Lambda'$ 都只剩第 $l$ 項：
$$\Omega(X_l^{-1}) = Y_l X_l\prod_{m \ne l}\Bigl(1 - \frac{X_m}{X_l}\Bigr),\qquad
\Lambda'(X_l^{-1}) = -X_l\prod_{m \ne l}\Bigl(1 - \frac{X_m}{X_l}\Bigr).$$
相除得到 **Forney formula**：
$$Y_l = -\frac{\Omega(X_l^{-1})}{\Lambda'(X_l^{-1})}.$$

在例子中，$S(z) = 2 + z + z^2 + 5z^3$，乘上 $\Lambda(z) = 1 + 5z + 4z^2$ 後只保留到 $z^3$，得到 $\Omega(z) = 2 + 4z$。  
又 $\Lambda'(z) = 5 + 8z = 5 + z$，而 $X_1^{-1} = 3^{-1} = 5$、$X_2^{-1} = 6^{-1} = 6$，所以
$$Y_1 = -\frac{\Omega(5)}{\Lambda'(5)} = -\frac13 = 2,\qquad
Y_2 = -\frac{\Omega(6)}{\Lambda'(6)} = -\frac54 = 4.$$

Key equation 的未知數 $\Lambda$ 與 $\Omega$ 都是低次多項式，有快速的解法：  
**Berlekamp–Massey algorithm** 與 **extended Euclidean algorithm**（Sugiyama 等人，1975），都只需要 $O(t^2)$ 次有限體運算。  
Peterson's method 每次嘗試都要解一個線性方程組，只適合小的 $t$。

### 兩種 decoders 的比較

| | Berlekamp–Welch | Syndrome decoding |
|---|---|---|
| 取值點 | 任意 $n$ 個不同的點 | $1, \alpha, \ldots, \alpha^{n - 1}$ |
| 輸入 | 全部 $n$ 個 received symbols | $n - k$ 個 syndromes |
| 主要計算 | $k + 2t$ 個未知數的線性方程組 | Key equation、Chien search、Forney formula |
| 輸出 | 資訊多項式 $f$ | Error vector $e$ |

Syndrome decoding 也能用在一般的取值點：改用 dual GRS code 給出的 parity-check matrix 計算 syndromes 即可。

## Bytes 與 Burst Errors

### 一個 symbol 是一個 byte

$\mathbb F_{256} = \mathbb F_{2^8}$ 的每個元素都是 $\mathbb F_2$ 上的 $8$ 維向量，也就是一個 byte。  
在 $\mathbb F_{256}$ 上，RS code 最長可以取 $n = 255$ 個非零元素當作取值點（加上 $0$ 則是 $256$ 個）。  
最有名的例子是 $[255, 223, 33]_{256}$：每 $223$ bytes 資訊加上 $32$ bytes parity，可以更正任意 $16$ 個 byte errors。

| 應用 | 使用的 RS codes |
|---|---|
| CD | 兩層 shortened RS codes $[32, 28, 5]_{256}$ 與 $[28, 24, 5]_{256}$，中間加上 interleaving |
| DVD | Product code，由 $[208, 192, 17]_{256}$ 與 $[182, 172, 11]_{256}$ 組成 |
| QR code | $\mathbb F_{256}$ 上的 RS codes，四種等級約可還原 $7\%$、$15\%$、$25\%$、$30\%$ 的 codewords |
| 深空通訊（CCSDS） | $[255, 223, 33]_{256}$ 作為外層，內層是 convolutional code |
| RAID 6 | 兩個 parity disks，容忍任意兩顆硬碟故障 |

### Shortened RS codes

CD 的 $[32, 28, 5]_{256}$ 長度不是 $255$。  
一種看法是直接只選 $32$ 個取值點。  
另一種看法是 **shortening**：在 systematic 的 $[255, 251, 5]_{256}$ 中，只取其中 $223$ 個資訊座標為零的 codewords，再刪掉這些座標。  
刪掉的座標全為零，weight 不變，所以 $d \ge 5$；而 Singleton bound 給出 $d \le 32 - 28 + 1 = 5$。  
兩種看法都得到 MDS code。

### Burst errors

許多通道的錯誤是成串出現的：光碟上的刮痕、儲存媒體的瑕疵、無線通道的短暫衰落。  
連續 $b$ 個 bits 的錯誤稱為長度 $b$ 的 **burst error**。  
RS decoder 只在乎有幾個 bytes 錯，不在乎每個 byte 錯了幾個 bits。  
長度 $b$ 的 burst 至多碰到
$$\left\lfloor\frac{b + 6}{8}\right\rfloor + 1$$
個 bytes；最壞情況是 burst 從某個 byte 的最後一個 bit 開始。

對 $[255, 223, 33]_{256}$，條件 $\lfloor(b + 6)/8\rfloor + 1 \le 16$ 等價於 $b \le 121$。  
所以任何長度至多 $121$ 的 burst 都能更正，即使它包含 $121$ 個 bit errors。  
若把同樣的 $2{,}040$ bits 當作 binary code，要更正任意 $121$ 個 bit errors，sphere packing bound 說至少需要約 $658$ bits 的 redundancy，遠多於這裡的 $256$ bits。

### 例子：在 $\mathbb F_{256}$ 上更正一個 burst

用 primitive polynomial $w^8 + w^4 + w^3 + w^2 + 1$ 建構 $\mathbb F_{256}$，這也是 QR code 使用的多項式。  
令 $\gamma$ 是 $w$ 的餘數類，它是 primitive element。  
用整數 $\sum_i b_i 2^i$ 標記元素 $\sum_i b_i\gamma^i$，$\mathbb F_{256}$ 的元素就和 bytes 一一對應。  
下面用 generator polynomial 做 systematic encoding，把 $223$ bytes 的文字編成 $[255, 223, 33]_{256}$ codeword。

In [10]:
W = PolynomialRing(GF(2), "w")
w = W.gen()
K = GF(2**8, "gamma", modulus=w**8 + w**4 + w**3 + w**2 + 1)
gamma = K.gen()
n, k = 255, 223
t = (n - k) // 2
points = [gamma**i for i in range(n)]
Kz = PolynomialRing(K, "z")
z = Kz.gen()
g = prod(z - gamma**j for j in range(1, n - k + 1))

In [11]:
message = b"Reed-Solomon codes treat each byte as one symbol of GF(256). "
message = (message * 4)[:k]
m = Kz([K.from_integer(byte) for byte in message])
shifted = z**(n - k) * m
c = vector(K, (shifted - shifted % g).padded_list(n))
print(bytes(s.to_integer() for s in c[n - k:]) == message)

True


接著把 codeword 攤平成 $2{,}040$ bits，從第 $999$ 個 bit 開始翻轉連續 $121$ 個 bits。  
$999$ 除以 $8$ 餘 $7$，所以 burst 從某個 byte 的最後一個 bit 開始，是最壞的情況。

In [12]:
bits = [int(bit) for s in c for bit in format(s.to_integer(), "08b")]
start, length = 999, 121
for i in range(start, start + length):
    bits[i] = 1 - bits[i]
received = vector(K, [K.from_integer(int("".join(map(str, bits[8*i:8*i + 8])), 2))
                      for i in range(n)])
print("bit errors:", length)
print("byte errors:", sum(1 for i in range(n) if received[i] != c[i]))

bit errors: 121
byte errors: 16


In [13]:
f_hat = bw_decode(points, received, k, t)
if f_hat is None:
    print("decoding failure")
else:
    decoded = vector(K, [f_hat(a) for a in points])
    print(decoded == c)
    print(bytes(s.to_integer() for s in decoded[n - k:]).decode())

True
Reed-Solomon codes treat each byte as one symbol of GF(256). Reed-Solomon codes treat each byte as one symbol of GF(256). Reed-Solomon codes treat each byte as one symbol of GF(256). Reed-Solomon codes treat each byte as on


Encoder 用的是 generator polynomial，decoder 用的是取值觀點的 Berlekamp–Welch；兩者能接起來，正是因為兩種描述給出同一個 code。  
$121$ 個 bit errors 只碰到 $16$ 個 bytes，恰好在 $t = 16$ 的範圍內。  
把 `length` 改成 $122$，burst 會碰到 $17$ 個 bytes，超出保證的範圍。

### Interleaving

更長的 burst，例如光碟上的大刮痕，會用 **interleaving** 處理。  
把 $L$ 個 codewords 排成 $L$ 個 rows，再按 column 的順序寫入媒體。  
一段長度至多 $L$ 個 symbols 的連續損壞，在每個 codeword 中至多造成一個 symbol error。  
CD 的兩層 RS codes 之間就有這樣的交錯：內層 decoder 標記為不可信的 symbols，到了外層變成分散的 erasures。

## 延伸話題

### Doubly extended RS 與 MDS conjecture

取值點也可以包含 $0$，得到 $n = q$。  
再加上一個「無窮遠點」的座標，記錄 $x^{k - 1}$ 的係數 $f_{k - 1}$：
$$f \longmapsto \bigl((f(a))_{a \in \mathbb F_q},\ f_{k - 1}\bigr).$$
這樣得到 $[q + 1, k, q - k + 2]_q$ 的 MDS code，稱為 **doubly extended RS code**。  
證明：若 $\deg f = k - 1$，最後一個座標不為零；若 $\deg f < k - 1$，$f$ 在 $\mathbb F_q$ 中至多有 $k - 2$ 個根。  
兩種情形的 weight 都至少是 $q - k + 2$。

**MDS conjecture** 猜測：$2 \le k \le q - 1$ 時，$\mathbb F_q$ 上 MDS codes 的長度不超過 $q + 1$。  
唯一的例外是 $q$ 為偶數且 $k = 3$ 或 $k = q - 1$，此時可以達到 $q + 2$。  
$q$ 是質數的情形已由 Ball（2012）證明。

### List decoding

Unique decoding 最多只能更正 $(n - k)/2$ 個 errors：再多就可能有兩個 codewords 一樣近。  
若允許 decoder 輸出一個很短的候選 list，Sudan（1997）與 Guruswami–Sudan（1999）的演算法可以在多項式時間內更正將近
$$n - \sqrt{nk}$$
個 errors。  
例如 rate $1/2$ 時，unique decoding 能更正約 $25\%$ 的 symbols，list decoding 則接近 $29\%$。  
它把 Berlekamp–Welch 的 $Q(x) - yE(x)$ 推廣成一般的二變數多項式 $Q(x, y)$ 的 interpolation。

## 週五作業

### 週一作業